In [1]:
# ============================================================
# M10_multiscale_dilated_cnn_regularized
# Smaller multi-scale CNN + stronger dropout + early stopping
# ============================================================

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from scipy.stats import pearsonr, spearmanr

# =========================
# MODEL NAME
# =========================

MODEL_NAME = "M10_multiscale_dilated_cnn_regularized"

# =========================
# SETTINGS
# =========================

TRAIN_PATH = "data/pep_nolog_sum_cf20_cf200_train.csv"
TEST_PATH  = "data/pep_nolog_sum_cf20_cf200_test_clean.csv"

SEQ_COLUMN = "peptide"
TARGET_COLUMN = "wash4"

MAX_LEN = 20
BATCH_SIZE = 128
EPOCHS = 50
PATIENCE = 5

LR = 5e-4
WEIGHT_DECAY = 5e-4

RANDOM_STATE = 42
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

OUT_DIR = MODEL_NAME
os.makedirs(OUT_DIR, exist_ok=True)

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

# =========================
# VOCABULARIES
# =========================

AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")
AA_TO_IDX = {aa: i + 1 for i, aa in enumerate(AA_LIST)}

AA_PAD = 0
AA_VOCAB_SIZE = len(AA_TO_IDX) + 1

# ============================================================
# NEW CHEMICAL CLASS SYSTEM
# G , A = Sm
# V, L, I, M, C, P = Hyd
# S, T, Q, N = Pol
# E, D, R, K, H = Cha
# W,F,Y = Aro
# ============================================================

CHEM_CLASSES = [
    "PAD",
    "Sm",
    "Hyd",
    "Pol",
    "Cha",
    "Aro",
    "Other"
]

CHEM_TO_IDX = {
    c: i for i, c in enumerate(CHEM_CLASSES)
}

def aa_to_chem(aa):

    # Small
    if aa in "GA":
        return "Sm"

    # Hydrophobic
    if aa in "VLIMCP":
        return "Hyd"

    # Polar
    if aa in "STQN":
        return "Pol"

    # Charged
    if aa in "EDRKH":
        return "Cha"

    # Aromatic
    if aa in "WFY":
        return "Aro"

    return "Other"

def encode_sequence(seq):

    seq = str(seq).upper().strip()

    aa_ids = np.zeros(MAX_LEN, dtype=np.int64)
    chem_ids = np.zeros(MAX_LEN, dtype=np.int64)

    for i, aa in enumerate(seq[:MAX_LEN]):

        aa_ids[i] = AA_TO_IDX.get(
            aa,
            AA_PAD
        )

        chem_ids[i] = CHEM_TO_IDX.get(
            aa_to_chem(aa),
            CHEM_TO_IDX["Other"]
        )

    return aa_ids, chem_ids

# =========================
# DATASET
# =========================

class PeptideDataset(Dataset):
    def __init__(self, df):
        self.seqs = (
            df[SEQ_COLUMN]
            .astype(str)
            .str.upper()
            .str.strip()
            .tolist()
        )

        self.y = df[TARGET_COLUMN].astype(float).values.astype(np.float32)

        aa_list = []
        chem_list = []

        for seq in self.seqs:
            aa_ids, chem_ids = encode_sequence(seq)
            aa_list.append(aa_ids)
            chem_list.append(chem_ids)

        self.aa_ids = np.array(aa_list)
        self.chem_ids = np.array(chem_list)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return {
            "aa_ids": torch.tensor(self.aa_ids[idx], dtype=torch.long),
            "chem_ids": torch.tensor(self.chem_ids[idx], dtype=torch.long),
            "y": torch.tensor(self.y[idx], dtype=torch.float32)
        }

# =========================
# MODEL
# =========================

class MultiScaleConvBranch(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, dilation, dropout=0.30):
        super().__init__()

        padding = ((kernel_size - 1) * dilation) // 2

        self.conv = nn.Conv1d(
            in_channels,
            out_channels,
            kernel_size=kernel_size,
            padding=padding,
            dilation=dilation
        )

        self.bn = nn.BatchNorm1d(out_channels)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        target_len = x.size(-1)

        out = self.conv(x)

        # Fix length mismatch for even kernels
        if out.size(-1) > target_len:
            out = out[:, :, :target_len]
        elif out.size(-1) < target_len:
            pad_amount = target_len - out.size(-1)
            out = torch.nn.functional.pad(out, (0, pad_amount))

        out = self.bn(out)
        out = self.relu(out)
        out = self.dropout(out)

        return out


class AttentionPooling(nn.Module):
    def __init__(self, hidden_dim):
        super().__init__()

        self.attn = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.Tanh(),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        weights = self.attn(x).squeeze(-1)
        weights = torch.softmax(weights, dim=1)
        pooled = torch.sum(x * weights.unsqueeze(-1), dim=1)
        return pooled, weights


class MultiScaleDilatedCNN(nn.Module):
    def __init__(
        self,
        aa_vocab_size,
        chem_vocab_size,
        aa_embed_dim=32,
        chem_embed_dim=16,
        branch_channels=32
    ):
        super().__init__()

        self.aa_embedding = nn.Embedding(
            aa_vocab_size,
            aa_embed_dim,
            padding_idx=0
        )

        self.chem_embedding = nn.Embedding(
            chem_vocab_size,
            chem_embed_dim,
            padding_idx=0
        )

        input_channels = aa_embed_dim + chem_embed_dim

        self.input_projection = nn.Conv1d(
            input_channels,
            branch_channels,
            kernel_size=1
        )

        self.branches = nn.ModuleList([
        # 3-position motifs
            MultiScaleConvBranch(branch_channels, branch_channels, kernel_size=3, dilation=1),
            MultiScaleConvBranch(branch_channels, branch_channels, kernel_size=3, dilation=2),
            MultiScaleConvBranch(branch_channels, branch_channels, kernel_size=3, dilation=4),
            MultiScaleConvBranch(branch_channels, branch_channels, kernel_size=3, dilation=5),

        # 4-position motifs
            MultiScaleConvBranch(branch_channels, branch_channels, kernel_size=4, dilation=1),
            MultiScaleConvBranch(branch_channels, branch_channels, kernel_size=4, dilation=2),
            MultiScaleConvBranch(branch_channels, branch_channels, kernel_size=4, dilation=3),
            MultiScaleConvBranch(branch_channels, branch_channels, kernel_size=4, dilation=4),
            MultiScaleConvBranch(branch_channels, branch_channels, kernel_size=4, dilation=5),

        # 5-position motifs
            MultiScaleConvBranch(branch_channels, branch_channels, kernel_size=5, dilation=1),
        ])

        total_channels = branch_channels * len(self.branches)

        self.fusion = nn.Sequential(
            nn.Conv1d(total_channels, 96, kernel_size=1),
            nn.BatchNorm1d(96),
            nn.ReLU(),
            nn.Dropout(0.30)
        )

        self.attention = AttentionPooling(96)

        self.regressor = nn.Sequential(
            nn.Linear(96, 96),
            nn.ReLU(),
            nn.Dropout(0.40),
            nn.Linear(96, 48),
            nn.ReLU(),
            nn.Dropout(0.30),
            nn.Linear(48, 1)
        )

    def forward(self, aa_ids, chem_ids):
        aa_emb = self.aa_embedding(aa_ids)
        chem_emb = self.chem_embedding(chem_ids)

        x = torch.cat([aa_emb, chem_emb], dim=-1)

        x = x.transpose(1, 2)

        x = self.input_projection(x)

        branch_outputs = [
            branch(x)
            for branch in self.branches
        ]

        x = torch.cat(branch_outputs, dim=1)

        x = self.fusion(x)

        x = x.transpose(1, 2)

        pooled, attn_weights = self.attention(x)

        out = self.regressor(pooled).squeeze(-1)

        return out, attn_weights

# =========================
# METRICS
# =========================

def safe_corr(y_true, y_pred):
    if len(y_true) < 3:
        return np.nan, np.nan

    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan, np.nan

    return (
        pearsonr(y_true, y_pred)[0],
        spearmanr(y_true, y_pred)[0]
    )

def compute_metrics(y_true, y_pred):
    pearson, spearman = safe_corr(y_true, y_pred)

    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Pearson": pearson,
        "Spearman": spearman
    }

def evaluate_model(model, loader):
    model.eval()

    all_y = []
    all_pred = []

    with torch.no_grad():
        for batch in loader:
            aa_ids = batch["aa_ids"].to(DEVICE)
            chem_ids = batch["chem_ids"].to(DEVICE)
            y = batch["y"].to(DEVICE)

            pred, _ = model(aa_ids, chem_ids)

            all_y.append(y.cpu().numpy())
            all_pred.append(pred.cpu().numpy())

    y_true = np.concatenate(all_y)
    y_pred = np.concatenate(all_pred)

    return y_true, y_pred, compute_metrics(y_true, y_pred)

def evaluate_subset(y_true, y_pred, subset_name):
    metrics = compute_metrics(y_true, y_pred)

    return {
        "model": MODEL_NAME,
        "subset": subset_name,
        "n_peptides": len(y_true),
        **metrics,
        "true_coam_mean": np.mean(y_true),
        "pred_coam_mean": np.mean(y_pred),
        "mean_underprediction": np.mean(y_true) - np.mean(y_pred)
    }

def evaluate_high_coam_subsets(y_true, y_pred):
    rows = []

    rows.append(
        evaluate_subset(y_true, y_pred, "Full test set")
    )

    for q, label in [
        (0.75, "Top 25% true coam"),
        (0.90, "Top 10% true coam"),
        (0.95, "Top 5% true coam")
    ]:
        threshold = np.quantile(y_true, q)
        mask = y_true >= threshold

        rows.append(
            evaluate_subset(y_true[mask], y_pred[mask], label)
        )

    return pd.DataFrame(rows)

# =========================
# LOAD DATA
# =========================

train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

for df in [train_df, test_df]:
    df.dropna(
        subset=[SEQ_COLUMN, TARGET_COLUMN],
        inplace=True
    )

    df[SEQ_COLUMN] = (
        df[SEQ_COLUMN]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df[TARGET_COLUMN] = pd.to_numeric(
        df[TARGET_COLUMN],
        errors="coerce"
    )

    df.dropna(
        subset=[TARGET_COLUMN],
        inplace=True
    )

    df.reset_index(drop=True, inplace=True)

train_dataset = PeptideDataset(train_df)
test_dataset = PeptideDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# =========================
# INIT MODEL
# =========================

model = MultiScaleDilatedCNN(
    aa_vocab_size=AA_VOCAB_SIZE,
    chem_vocab_size=len(CHEM_CLASSES)
).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

loss_fn = nn.MSELoss(reduction="none")

best_test_r2 = -999
best_epoch = 0
epochs_without_improvement = 0

best_path = os.path.join(
    OUT_DIR,
    f"{MODEL_NAME}_best.pt"
)

print("==========================================================")
print(MODEL_NAME)
print("==========================================================")
print("Device:", DEVICE)
print("Train size:", len(train_dataset))
print("Test size :", len(test_dataset))
print("Architecture: smaller multiscale CNN")
print("Branches: k3-d1, k3-d2, k3-d4, k3-d5, k4-d1, k4-d2, k4-d3, k4-d4, k4-d5, k5-d1")
print("Early stopping patience:", PATIENCE)

# =========================
# TRAIN LOOP WITH EARLY STOPPING
# =========================

for epoch in range(1, EPOCHS + 1):
    model.train()

    losses_epoch = []

    for batch in train_loader:
        aa_ids = batch["aa_ids"].to(DEVICE)
        chem_ids = batch["chem_ids"].to(DEVICE)
        y = batch["y"].to(DEVICE)

        optimizer.zero_grad()

        pred, _ = model(aa_ids, chem_ids)

        losses = loss_fn(pred, y)

        sample_weights = 1.0 + 3.0 * y

        loss = (losses * sample_weights).mean()

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        losses_epoch.append(loss.item())

    train_y, train_pred, train_metrics = evaluate_model(model, train_loader)
    test_y, test_pred, test_metrics = evaluate_model(model, test_loader)

    improved = test_metrics["R2"] > best_test_r2

    if improved:
        best_test_r2 = test_metrics["R2"]
        best_epoch = epoch
        epochs_without_improvement = 0
        torch.save(model.state_dict(), best_path)
    else:
        epochs_without_improvement += 1

    print(
        f"Epoch {epoch:03d} | "
        f"Loss {np.mean(losses_epoch):.6f} | "
        f"Train R2 {train_metrics['R2']:.4f} | "
        f"Test R2 {test_metrics['R2']:.4f} | "
        f"Test MAE {test_metrics['MAE']:.5f} | "
        f"Test Pearson {test_metrics['Pearson']:.4f} | "
        f"Best epoch {best_epoch}"
    )

    if epochs_without_improvement >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}. Best epoch: {best_epoch}")
        break

# =========================
# FINAL EVALUATION
# =========================

model.load_state_dict(
    torch.load(best_path, map_location=DEVICE)
)

train_y, train_pred, train_metrics = evaluate_model(model, train_loader)
test_y, test_pred, test_metrics = evaluate_model(model, test_loader)

print("\nBest model train metrics:")
for k, v in train_metrics.items():
    print(f"{k}: {v:.4f}")

print("\nBest model test metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")

eval_df = evaluate_high_coam_subsets(test_y, test_pred)

print("\nHigh-coam subset evaluation:")
print(eval_df.to_string(index=False))

eval_df.to_csv(
    f"{MODEL_NAME}_high_coam_eval.csv",
    index=False
)

predictions_df = pd.DataFrame({
    "model": MODEL_NAME,
    SEQ_COLUMN: test_df[SEQ_COLUMN].values,
    "true_coam": test_y,
    "predicted_coam": test_pred,
    "residual_true_minus_pred": test_y - test_pred
})

predictions_df.to_csv(
    f"{MODEL_NAME}_predictions.csv",
    index=False
)

summary_df = pd.DataFrame([
    {
        "model": MODEL_NAME,
        "dataset": "train",
        "best_epoch": best_epoch,
        **train_metrics
    },
    {
        "model": MODEL_NAME,
        "dataset": "test",
        "best_epoch": best_epoch,
        **test_metrics
    }
])

summary_df.to_csv(
    f"{MODEL_NAME}_train_test_summary.csv",
    index=False
)

print("\nSaved:")
print(best_path)
print(f"{MODEL_NAME}_high_coam_eval.csv")
print(f"{MODEL_NAME}_predictions.csv")
print(f"{MODEL_NAME}_train_test_summary.csv")

M10_multiscale_dilated_cnn_regularized
Device: cuda
Train size: 58579
Test size : 5119
Architecture: smaller multiscale CNN
Branches: k3-d1, k3-d2, k3-d4, k3-d5, k4-d1, k4-d2, k4-d3, k4-d4, k4-d5, k5-d1
Early stopping patience: 5
Epoch 001 | Loss 0.032752 | Train R2 0.0956 | Test R2 0.0383 | Test MAE 0.09285 | Test Pearson 0.4135 | Best epoch 1
Epoch 002 | Loss 0.019348 | Train R2 0.2856 | Test R2 0.1824 | Test MAE 0.08372 | Test Pearson 0.4333 | Best epoch 2
Epoch 003 | Loss 0.016460 | Train R2 0.2809 | Test R2 0.1768 | Test MAE 0.08425 | Test Pearson 0.4396 | Best epoch 2
Epoch 004 | Loss 0.014962 | Train R2 0.3063 | Test R2 0.1966 | Test MAE 0.08260 | Test Pearson 0.4447 | Best epoch 4
Epoch 005 | Loss 0.014161 | Train R2 0.3248 | Test R2 0.2054 | Test MAE 0.08213 | Test Pearson 0.4533 | Best epoch 5
Epoch 006 | Loss 0.013546 | Train R2 0.3332 | Test R2 0.2073 | Test MAE 0.08189 | Test Pearson 0.4556 | Best epoch 6
Epoch 007 | Loss 0.013128 | Train R2 0.3369 | Test R2 0.2040 | Test 

In [3]:
# ============================================================
# M04_onehot_engineered_xgboost_gpu
# XGBoost GPU model using ONE-HOT + ENGINEERED FEATURES
# ============================================================

import pandas as pd
import numpy as np
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor
import torch

# =========================
# MODEL NAME
# =========================

MODEL_NAME = "M04_onehot_engineered_xgboost_gpu"

# =========================
# SETTINGS
# =========================

TRAIN_PATH = "data/pep_nolog_sum_cf20_cf200_train.csv"
TEST_PATH  = "data/pep_nolog_sum_cf20_cf200_test_clean.csv"

SEQ_COLUMN = "peptide"
TARGET_COLUMN = "wash4"

MAX_LEN = 12
RANDOM_STATE = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# ONE-HOT ENCODING
# =========================

AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")
AA_TO_INDEX = {aa: i for i, aa in enumerate(AA_LIST)}

onehot_feature_names = [
    f"OH_pos{i+1}_{aa}"
    for i in range(MAX_LEN)
    for aa in AA_LIST
]

def encode_onehot(seq):
    seq = str(seq).upper().strip()

    x = np.zeros(
        MAX_LEN * len(AA_LIST),
        dtype=float
    )

    for i in range(min(len(seq), MAX_LEN)):
        aa = seq[i]

        if aa in AA_TO_INDEX:
            x[i * len(AA_LIST) + AA_TO_INDEX[aa]] = 1.0

    return x

# =========================
# ENGINEERED FEATURES
# =========================

SELECTED_FEATURES = [
    "charge_balance",
    "sulfur_fraction",
    "local_polarizability_patch",
    "max_aromatic_run",
    "F_fraction",
    "aromatic_fraction",
    "W_positive_contacts",
    "CC_contact_score",
    "aromatic_positive_contacts",
    "pi_density",
    "C_fraction",
    "aromatic_charged_contacts",
    "aromatic_cluster_score",
    "has_terminal_C",
    "adjacent_aromatic_pairs",
    "C_aromatic_contact_score",
    "aromatic_polar_contacts",
    "aromatic_sulfur_contacts",
    "terminal_aromatic_score",
    "W_fraction",
    "Y_fraction",
    "aromatic_hydrophobic_fraction",
    "Nterm_pi_density",
    "W_C_contacts",
    "terminal_WC_score",
    "pi_sulfur_score",
    "C_positive_contact_score",
    "M_fraction",
    "alpha_propensity",
    "beta_propensity",
    "flexibility",
    "Nterm_flexibility",
    "Cterm_flexibility",
]

AROMATIC = set("WFY")
AROMATIC_EXTENDED = set("WFYH")
SULFUR = set("CM")
POSITIVE = set("RKH")
NEGATIVE = set("DE")
CHARGED = POSITIVE | NEGATIVE
POLAR = set("STNQ")
HYDROPHOBIC = set("VILMCPA")

PI_WEIGHTS = {"W": 3.0, "Y": 2.5, "F": 2.0, "H": 1.5}

POLARIZABILITY_WEIGHTS = {
    "W": 3.0,
    "Y": 2.5,
    "F": 2.0,
    "H": 1.7,
    "C": 2.2,
    "M": 1.8
}

ALPHA_PROPENSITY = {
    "A": 1.45, "R": 0.79, "N": 0.73, "D": 0.98,
    "C": 0.77, "Q": 1.17, "E": 1.53, "G": 0.53,
    "H": 1.24, "I": 1.00, "L": 1.34, "K": 1.07,
    "M": 1.20, "F": 1.12, "P": 0.59, "S": 0.79,
    "T": 0.82, "W": 1.14, "Y": 0.61, "V": 1.14
}

BETA_PROPENSITY = {
    "A": 0.97, "R": 0.90, "N": 0.65, "D": 0.80,
    "C": 1.30, "Q": 1.23, "E": 0.26, "G": 0.81,
    "H": 0.71, "I": 1.60, "L": 1.22, "K": 0.74,
    "M": 1.67, "F": 1.28, "P": 0.62, "S": 0.72,
    "T": 1.20, "W": 1.19, "Y": 1.29, "V": 1.65
}

FLEXIBILITY = {
    "A": 0.36, "R": 0.53, "N": 0.46, "D": 0.51,
    "C": 0.35, "Q": 0.49, "E": 0.50, "G": 0.54,
    "H": 0.32, "I": 0.46, "L": 0.37, "K": 0.47,
    "M": 0.30, "F": 0.31, "P": 0.51, "S": 0.51,
    "T": 0.44, "W": 0.31, "Y": 0.42, "V": 0.39
}

def count_group(seq, group):
    return sum(aa in group for aa in seq)

def max_run(seq, group):
    best = 0
    current = 0

    for aa in seq:
        if aa in group:
            current += 1
            best = max(best, current)
        else:
            current = 0

    return best

def pair_contacts(seq, group1, group2, max_dist=2):
    count = 0

    for i, aa1 in enumerate(seq):
        if aa1 not in group1:
            continue

        for j in range(
            max(0, i - max_dist),
            min(len(seq), i + max_dist + 1)
        ):
            if i != j and seq[j] in group2:
                count += 1

    return count

def nearby_count(pos1, pos2, max_dist=2):
    count = 0

    for i in pos1:
        for j in pos2:
            if i != j and abs(i - j) <= max_dist:
                count += 1

    return count

def weighted_density(seq, weights):
    L = max(len(seq), 1)

    return sum(
        weights.get(aa, 0)
        for aa in seq
    ) / L

def local_weighted_cluster(seq, weights, window=2):
    if len(seq) == 0:
        return 0

    scores = []

    for i in range(len(seq)):
        start = max(0, i - window)
        end = min(len(seq), i + window + 1)

        score = sum(
            weights.get(aa, 0)
            for aa in seq[start:end]
        )

        scores.append(score)

    return max(scores)

def terminal_score(seq, group, n=3):
    if len(seq) == 0:
        return 0

    return (
        count_group(seq[:n], group)
        +
        count_group(seq[-n:], group)
    )

def calculate_engineered_features(seq):
    seq = str(seq).upper().strip()

    L = max(len(seq), 1)

    aro_count = count_group(seq, AROMATIC)
    sulfur_count = count_group(seq, SULFUR)

    pos_count = count_group(seq, POSITIVE)
    neg_count = count_group(seq, NEGATIVE)

    c_count = seq.count("C")

    C_positions = [
        i for i, aa in enumerate(seq)
        if aa == "C"
    ]

    aromatic_positions = [
        i for i, aa in enumerate(seq)
        if aa in AROMATIC_EXTENDED
    ]

    positive_positions = [
        i for i, aa in enumerate(seq)
        if aa in POSITIVE
    ]

    d = {
        "pi_density":
            weighted_density(seq, PI_WEIGHTS),

        "aromatic_fraction":
            aro_count / L,

        "W_fraction":
            seq.count("W") / L,

        "Y_fraction":
            seq.count("Y") / L,

        "F_fraction":
            seq.count("F") / L,

        "max_aromatic_run":
            max_run(seq, AROMATIC),

        "aromatic_cluster_score":
            pair_contacts(seq, AROMATIC, AROMATIC, 2) / L,

        "adjacent_aromatic_pairs":
            pair_contacts(seq, AROMATIC, AROMATIC, 1) / L,

        "sulfur_fraction":
            sulfur_count / L,

        "C_fraction":
            c_count / L,

        "M_fraction":
            seq.count("M") / L,

        "local_polarizability_patch":
            local_weighted_cluster(seq, POLARIZABILITY_WEIGHTS, 2),

        "aromatic_sulfur_contacts":
            pair_contacts(seq, AROMATIC, SULFUR, 2) / L,

        "W_C_contacts":
            pair_contacts(seq, set("W"), set("C"), 2) / L,

        "pi_sulfur_score":
            weighted_density(seq, PI_WEIGHTS) * sulfur_count / L,

        "aromatic_positive_contacts":
            pair_contacts(seq, AROMATIC, POSITIVE, 2) / L,

        "aromatic_charged_contacts":
            pair_contacts(seq, AROMATIC, CHARGED, 2) / L,

        "W_positive_contacts":
            pair_contacts(seq, set("W"), POSITIVE, 2) / L,

        "charge_balance":
            (pos_count - neg_count) / L,

        "aromatic_polar_contacts":
            pair_contacts(seq, AROMATIC, POLAR, 2) / L,

        "aromatic_hydrophobic_fraction":
            count_group(seq, AROMATIC | HYDROPHOBIC) / L,

        "terminal_aromatic_score":
            terminal_score(seq, AROMATIC, 3),

        "terminal_WC_score":
            terminal_score(seq, set("WC"), 3),

        "Nterm_pi_density":
            weighted_density(seq[:3], PI_WEIGHTS),

        "has_terminal_C":
            int("C" in seq[:3] or "C" in seq[-3:]),

        "C_aromatic_contact_score":
            nearby_count(C_positions, aromatic_positions, 2) / L,

        "C_positive_contact_score":
            nearby_count(C_positions, positive_positions, 2) / L,

        "CC_contact_score":
            nearby_count(C_positions, C_positions, 2) / L,

        "alpha_propensity":
            weighted_density(seq, ALPHA_PROPENSITY),

        "beta_propensity":
            weighted_density(seq, BETA_PROPENSITY),

        "flexibility":
            weighted_density(seq, FLEXIBILITY),

        "Nterm_flexibility":
            weighted_density(seq[:3], FLEXIBILITY),

        "Cterm_flexibility":
            weighted_density(seq[-3:], FLEXIBILITY),
    }

    return [
        d[f]
        for f in SELECTED_FEATURES
    ]

# =========================
# LOAD DATA
# =========================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

for df in [train, test]:

    df.dropna(
        subset=[SEQ_COLUMN, TARGET_COLUMN],
        inplace=True
    )

    df[SEQ_COLUMN] = (
        df[SEQ_COLUMN]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df[TARGET_COLUMN] = pd.to_numeric(
        df[TARGET_COLUMN],
        errors="coerce"
    )

    df.dropna(
        subset=[TARGET_COLUMN],
        inplace=True
    )

    df.reset_index(
        drop=True,
        inplace=True
    )

y_train = train[TARGET_COLUMN].values
y_test = test[TARGET_COLUMN].values

# =========================
# BUILD MATRICES
# =========================

X_onehot_train = np.array([
    encode_onehot(seq)
    for seq in train[SEQ_COLUMN]
])

X_onehot_test = np.array([
    encode_onehot(seq)
    for seq in test[SEQ_COLUMN]
])

X_engineered_train = np.array([
    calculate_engineered_features(seq)
    for seq in train[SEQ_COLUMN]
])

X_engineered_test = np.array([
    calculate_engineered_features(seq)
    for seq in test[SEQ_COLUMN]
])

X_train = np.hstack([
    X_onehot_train,
    X_engineered_train
])

X_test = np.hstack([
    X_onehot_test,
    X_engineered_test
])

feature_names = (
    onehot_feature_names
    +
    SELECTED_FEATURES
)

print("==========================================================")
print(MODEL_NAME)
print("==========================================================")
print("Feature type: one-hot + engineered")
print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
print("One-hot features:", len(onehot_feature_names))
print("Engineered features:", len(SELECTED_FEATURES))
print("Total features:", len(feature_names))
print("Device:", DEVICE)

# =========================
# TRAIN MODEL
# =========================

model = XGBRegressor(
    n_estimators=1500,
    max_depth=6,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    device="cuda" if DEVICE == "cuda" else "cpu",
    random_state=RANDOM_STATE
)

model.fit(
    X_train,
    y_train
)

# =========================
# PREDICTIONS
# =========================

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# =========================
# METRICS
# =========================

def safe_corr(y_true, y_pred):
    if len(y_true) < 3:
        return np.nan, np.nan

    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan, np.nan

    return (
        pearsonr(y_true, y_pred)[0],
        spearmanr(y_true, y_pred)[0]
    )

def metrics(y_true, y_pred):
    pearson, spearman = safe_corr(
        y_true,
        y_pred
    )

    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Pearson": pearson,
        "Spearman": spearman
    }

train_metrics = metrics(
    y_train,
    train_pred
)

test_metrics = metrics(
    y_test,
    test_pred
)

print("\nTrain metrics:")
for k, v in train_metrics.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")

# =========================
# HIGH-COAM SUBSET EVALUATION
# =========================

def evaluate_subset(y_true, y_pred, subset_name):
    pearson, spearman = safe_corr(
        y_true,
        y_pred
    )

    return {
        "model": MODEL_NAME,
        "subset": subset_name,
        "n_peptides": len(y_true),
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Pearson": pearson,
        "Spearman": spearman,
        "true_coam_mean": np.mean(y_true),
        "pred_coam_mean": np.mean(y_pred),
        "mean_underprediction": np.mean(y_true) - np.mean(y_pred)
    }

eval_rows = []

eval_rows.append(
    evaluate_subset(
        y_test,
        test_pred,
        "Full test set"
    )
)

for q, label in [
    (0.75, "Top 25% true coam"),
    (0.90, "Top 10% true coam"),
    (0.95, "Top 5% true coam")
]:

    threshold = np.quantile(
        y_test,
        q
    )

    mask = y_test >= threshold

    eval_rows.append(
        evaluate_subset(
            y_test[mask],
            test_pred[mask],
            label
        )
    )

eval_df = pd.DataFrame(eval_rows)

print("\nHigh-coam subset evaluation:")
print(
    eval_df.to_string(index=False)
)

eval_df.to_csv(
    f"{MODEL_NAME}_high_coam_eval.csv",
    index=False
)

# =========================
# SAVE MODEL
# =========================

model.save_model(
    f"{MODEL_NAME}.json"
)

print(f"\nSaved model: {MODEL_NAME}.json")

# =========================
# FEATURE IMPORTANCE
# =========================

importance_df = pd.DataFrame({
    "model": MODEL_NAME,
    "feature": feature_names,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

importance_df.to_csv(
    f"{MODEL_NAME}_feature_importances.csv",
    index=False
)

print("\nTop 30 feature importances:")
print(
    importance_df
    .head(30)
    .to_string(index=False)
)

# =========================
# SAVE PREDICTIONS
# =========================

predictions_df = pd.DataFrame({
    "model": MODEL_NAME,
    SEQ_COLUMN: test[SEQ_COLUMN].values,
    "true_coam": y_test,
    "predicted_coam": test_pred,
    "residual_true_minus_pred": y_test - test_pred
})

predictions_df.to_csv(
    f"{MODEL_NAME}_predictions.csv",
    index=False
)

# =========================
# SAVE TRAIN/TEST SUMMARY
# =========================

summary_df = pd.DataFrame([
    {
        "model": MODEL_NAME,
        "dataset": "train",
        **train_metrics
    },
    {
        "model": MODEL_NAME,
        "dataset": "test",
        **test_metrics
    }
])

summary_df.to_csv(
    f"{MODEL_NAME}_train_test_summary.csv",
    index=False
)

print("\nSaved:")
print(f"{MODEL_NAME}_high_coam_eval.csv")
print(f"{MODEL_NAME}_feature_importances.csv")
print(f"{MODEL_NAME}_predictions.csv")
print(f"{MODEL_NAME}_train_test_summary.csv")

M04_onehot_engineered_xgboost_gpu
Feature type: one-hot + engineered
Train shape: (58579, 273)
Test shape : (5119, 273)
One-hot features: 240
Engineered features: 33
Total features: 273
Device: cuda

Train metrics:
R2: 0.6186
MAE: 0.0443
RMSE: 0.0606
Pearson: 0.8060
Spearman: 0.7788

Test metrics:
R2: 0.2029
MAE: 0.0824
RMSE: 0.1080
Pearson: 0.4509
Spearman: 0.4326

High-coam subset evaluation:
                            model            subset  n_peptides         R2      MAE     RMSE  Pearson  Spearman  true_coam_mean  pred_coam_mean  mean_underprediction
M04_onehot_engineered_xgboost_gpu     Full test set        5119   0.202891 0.082366 0.107957 0.450934  0.432557        0.313862        0.311368              0.002495
M04_onehot_engineered_xgboost_gpu Top 25% true coam        1280  -2.713701 0.119177 0.138529 0.393033  0.359048        0.468338        0.351563              0.116775
M04_onehot_engineered_xgboost_gpu Top 10% true coam         512  -6.663051 0.164133 0.180720 0.314836  0

In [4]:
# ============================================================
# ONE-HOT ENCODING + GPU XGBOOST REGRESSION MODEL
# ============================================================

import pandas as pd
import numpy as np
import torch

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

# =========================
# MODEL NAME
# =========================

MODEL_NAME = "M_onehot_xgboost_gpu"

# =========================
# SETTINGS
# =========================

TRAIN_PATH = "data/pep_nolog_sum_cf20_cf200_train.csv"
TEST_PATH  = "data/pep_nolog_sum_cf20_cf200_test_clean.csv"

SEQ_COLUMN = "peptide"
TARGET_COLUMN = "wash4"

MAX_LEN = 12
RANDOM_STATE = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# ONE-HOT ENCODING
# =========================

AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")

AA_TO_INDEX = {
    aa: i
    for i, aa in enumerate(AA_LIST)
}

onehot_feature_names = [
    f"OH_pos{i+1}_{aa}"
    for i in range(MAX_LEN)
    for aa in AA_LIST
]

def encode_onehot(seq):

    seq = str(seq).upper().strip()

    x = np.zeros(
        MAX_LEN * len(AA_LIST),
        dtype=float
    )

    for i in range(min(len(seq), MAX_LEN)):

        aa = seq[i]

        if aa in AA_TO_INDEX:

            x[
                i * len(AA_LIST)
                +
                AA_TO_INDEX[aa]
            ] = 1.0

    return x

# =========================
# LOAD DATA
# =========================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

for df in [train, test]:

    df.dropna(
        subset=[SEQ_COLUMN, TARGET_COLUMN],
        inplace=True
    )

    df[SEQ_COLUMN] = (
        df[SEQ_COLUMN]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df[TARGET_COLUMN] = pd.to_numeric(
        df[TARGET_COLUMN],
        errors="coerce"
    )

    df.dropna(
        subset=[TARGET_COLUMN],
        inplace=True
    )

    df.reset_index(
        drop=True,
        inplace=True
    )

y_train = train[TARGET_COLUMN].values
y_test = test[TARGET_COLUMN].values

# =========================
# BUILD ONE-HOT MATRICES
# =========================

X_train = np.array([
    encode_onehot(seq)
    for seq in train[SEQ_COLUMN]
])

X_test = np.array([
    encode_onehot(seq)
    for seq in test[SEQ_COLUMN]
])

feature_names = onehot_feature_names

print("==========================================================")
print(MODEL_NAME)
print("==========================================================")
print("Feature type: One-hot encoding only")
print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
print("Total one-hot features:", len(feature_names))
print("Device:", DEVICE)

# =========================
# TRAIN GPU XGBOOST MODEL
# =========================

model = XGBRegressor(
    n_estimators=1500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    device="cuda" if DEVICE == "cuda" else "cpu",
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=RANDOM_STATE
)

model.fit(
    X_train,
    y_train
)

# =========================
# PREDICTIONS
# =========================

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# =========================
# METRICS
# =========================

def safe_corr(y_true, y_pred):

    if len(y_true) < 3:
        return np.nan, np.nan

    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan, np.nan

    return (
        pearsonr(y_true, y_pred)[0],
        spearmanr(y_true, y_pred)[0]
    )

def metrics(y_true, y_pred):

    pearson, spearman = safe_corr(
        y_true,
        y_pred
    )

    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Pearson": pearson,
        "Spearman": spearman
    }

train_metrics = metrics(
    y_train,
    train_pred
)

test_metrics = metrics(
    y_test,
    test_pred
)

print("\nTrain metrics:")
for k, v in train_metrics.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")

# =========================
# HIGH-COAM SUBSET EVALUATION
# =========================

def evaluate_subset(y_true, y_pred, subset_name):

    pearson, spearman = safe_corr(
        y_true,
        y_pred
    )

    return {
        "model": MODEL_NAME,
        "subset": subset_name,
        "n_peptides": len(y_true),
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Pearson": pearson,
        "Spearman": spearman,
        "true_coam_mean": np.mean(y_true),
        "pred_coam_mean": np.mean(y_pred),
        "mean_underprediction": np.mean(y_true) - np.mean(y_pred)
    }

eval_rows = []

eval_rows.append(
    evaluate_subset(
        y_test,
        test_pred,
        "Full test set"
    )
)

for q, label in [
    (0.75, "Top 25% true coam"),
    (0.90, "Top 10% true coam"),
    (0.95, "Top 5% true coam")
]:

    threshold = np.quantile(
        y_test,
        q
    )

    mask = y_test >= threshold

    eval_rows.append(
        evaluate_subset(
            y_test[mask],
            test_pred[mask],
            label
        )
    )

eval_df = pd.DataFrame(eval_rows)

print("\nHigh-coam subset evaluation:")
print(
    eval_df.to_string(index=False)
)

eval_df.to_csv(
    f"{MODEL_NAME}_high_coam_eval.csv",
    index=False
)

# =========================
# SAVE MODEL
# =========================

model.save_model(
    f"{MODEL_NAME}.json"
)

print(f"\nSaved model: {MODEL_NAME}.json")

# =========================
# FEATURE IMPORTANCE
# =========================

importance_df = pd.DataFrame({
    "model": MODEL_NAME,
    "feature": feature_names,
    "importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

importance_df.to_csv(
    f"{MODEL_NAME}_feature_importances.csv",
    index=False
)

print("\nTop 30 feature importances:")
print(
    importance_df
    .head(30)
    .to_string(index=False)
)

# =========================
# SAVE PREDICTIONS
# =========================

predictions_df = pd.DataFrame({
    "model": MODEL_NAME,
    SEQ_COLUMN: test[SEQ_COLUMN].values,
    "true_coam": y_test,
    "predicted_coam": test_pred,
    "residual_true_minus_pred": y_test - test_pred
})

predictions_df.to_csv(
    f"{MODEL_NAME}_predictions.csv",
    index=False
)

# =========================
# SAVE SUMMARY
# =========================

summary_df = pd.DataFrame([
    {
        "model": MODEL_NAME,
        "dataset": "train",
        **train_metrics
    },
    {
        "model": MODEL_NAME,
        "dataset": "test",
        **test_metrics
    }
])

summary_df.to_csv(
    f"{MODEL_NAME}_train_test_summary.csv",
    index=False
)

print("\nSaved:")
print(f"{MODEL_NAME}_high_coam_eval.csv")
print(f"{MODEL_NAME}_feature_importances.csv")
print(f"{MODEL_NAME}_predictions.csv")
print(f"{MODEL_NAME}_train_test_summary.csv")

M_onehot_xgboost_gpu
Feature type: One-hot encoding only
Train shape: (58579, 240)
Test shape : (5119, 240)
Total one-hot features: 240
Device: cuda

Train metrics:
R2: 0.7316
MAE: 0.0372
RMSE: 0.0508
Pearson: 0.8827
Spearman: 0.8674

Test metrics:
R2: 0.1564
MAE: 0.0856
RMSE: 0.1111
Pearson: 0.3965
Spearman: 0.3821

High-coam subset evaluation:
               model            subset  n_peptides         R2      MAE     RMSE  Pearson  Spearman  true_coam_mean  pred_coam_mean  mean_underprediction
M_onehot_xgboost_gpu     Full test set        5119   0.156433 0.085594 0.111058 0.396461  0.382103        0.313862        0.312011              0.001851
M_onehot_xgboost_gpu Top 25% true coam        1280  -3.122607 0.126597 0.145956 0.318550  0.297379        0.468338        0.343609              0.124729
M_onehot_xgboost_gpu Top 10% true coam         512  -7.664707 0.176778 0.192169 0.261530  0.248070        0.537237        0.360535              0.176702
M_onehot_xgboost_gpu  Top 5% true coam  

In [6]:
# ============================================================
# ONE-HOT AA + CHEMICAL CLASS + GPU XGBOOST REGRESSION MODEL
# ============================================================

import pandas as pd
import numpy as np
import torch

from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

# =========================
# MODEL NAME
# =========================

MODEL_NAME = "M_onehot20_chemclass5_xgboost_gpu"

# =========================
# SETTINGS
# =========================

TRAIN_PATH = "data/pep_nolog_sum_cf20_cf200_train.csv"
TEST_PATH  = "data/pep_nolog_sum_cf20_cf200_test_clean.csv"

SEQ_COLUMN = "peptide"
TARGET_COLUMN = "wash4"

MAX_LEN = 12
RANDOM_STATE = 42

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# =========================
# AA ONE-HOT + CHEMICAL CLASS ONE-HOT
# =========================

AA_LIST = list("ACDEFGHIKLMNPQRSTVWY")

AA_TO_INDEX = {
    aa: i for i, aa in enumerate(AA_LIST)
}

CHEM_CLASSES = ["Sm", "Hyd", "Pol", "Cha", "Aro"]

CHEM_TO_INDEX = {
    chem: i for i, chem in enumerate(CHEM_CLASSES)
}

def aa_to_chem(aa):
    if aa in "GA":
        return "Sm"
    elif aa in "VLIMCP":
        return "Hyd"
    elif aa in "STQN":
        return "Pol"
    elif aa in "EDRKH":
        return "Cha"
    elif aa in "WFY":
        return "Aro"
    else:
        return None

aa_feature_names = [
    f"AA_OH_pos{i+1}_{aa}"
    for i in range(MAX_LEN)
    for aa in AA_LIST
]

chem_feature_names = [
    f"CHEM_OH_pos{i+1}_{chem}"
    for i in range(MAX_LEN)
    for chem in CHEM_CLASSES
]

feature_names = aa_feature_names + chem_feature_names

def encode_onehot_aa_plus_chem(seq):

    seq = str(seq).upper().strip()

    aa_x = np.zeros(
        MAX_LEN * len(AA_LIST),
        dtype=float
    )

    chem_x = np.zeros(
        MAX_LEN * len(CHEM_CLASSES),
        dtype=float
    )

    for i in range(min(len(seq), MAX_LEN)):

        aa = seq[i]

        if aa in AA_TO_INDEX:
            aa_x[
                i * len(AA_LIST)
                +
                AA_TO_INDEX[aa]
            ] = 1.0

        chem = aa_to_chem(aa)

        if chem in CHEM_TO_INDEX:
            chem_x[
                i * len(CHEM_CLASSES)
                +
                CHEM_TO_INDEX[chem]
            ] = 1.0

    return np.concatenate([aa_x, chem_x])

# =========================
# LOAD DATA
# =========================

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

for df in [train, test]:

    df.dropna(
        subset=[SEQ_COLUMN, TARGET_COLUMN],
        inplace=True
    )

    df[SEQ_COLUMN] = (
        df[SEQ_COLUMN]
        .astype(str)
        .str.upper()
        .str.strip()
    )

    df[TARGET_COLUMN] = pd.to_numeric(
        df[TARGET_COLUMN],
        errors="coerce"
    )

    df.dropna(
        subset=[TARGET_COLUMN],
        inplace=True
    )

    df.reset_index(
        drop=True,
        inplace=True
    )

y_train = train[TARGET_COLUMN].values
y_test = test[TARGET_COLUMN].values

# =========================
# BUILD FEATURE MATRICES
# =========================

X_train = np.array([
    encode_onehot_aa_plus_chem(seq)
    for seq in train[SEQ_COLUMN]
])

X_test = np.array([
    encode_onehot_aa_plus_chem(seq)
    for seq in test[SEQ_COLUMN]
])

print("==========================================================")
print(MODEL_NAME)
print("==========================================================")
print("Feature type: AA one-hot 20 + chemical class one-hot 5")
print("Train shape:", X_train.shape)
print("Test shape :", X_test.shape)
print("AA one-hot features:", len(aa_feature_names))
print("Chemical class features:", len(chem_feature_names))
print("Total features:", len(feature_names))
print("Device:", DEVICE)

# =========================
# TRAIN GPU XGBOOST MODEL
# =========================

model = XGBRegressor(
    n_estimators=1500,
    max_depth=8,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    device="cuda" if DEVICE == "cuda" else "cpu",
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=RANDOM_STATE
)

model.fit(
    X_train,
    y_train
)

# =========================
# PREDICTIONS
# =========================

train_pred = model.predict(X_train)
test_pred = model.predict(X_test)

# =========================
# METRICS
# =========================

def safe_corr(y_true, y_pred):

    if len(y_true) < 3:
        return np.nan, np.nan

    if np.std(y_true) == 0 or np.std(y_pred) == 0:
        return np.nan, np.nan

    return (
        pearsonr(y_true, y_pred)[0],
        spearmanr(y_true, y_pred)[0]
    )

def metrics(y_true, y_pred):

    pearson, spearman = safe_corr(
        y_true,
        y_pred
    )

    return {
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Pearson": pearson,
        "Spearman": spearman
    }

train_metrics = metrics(y_train, train_pred)
test_metrics = metrics(y_test, test_pred)

print("\nTrain metrics:")
for k, v in train_metrics.items():
    print(f"{k}: {v:.4f}")

print("\nTest metrics:")
for k, v in test_metrics.items():
    print(f"{k}: {v:.4f}")

# =========================
# HIGH-COAM SUBSET EVALUATION
# =========================

def evaluate_subset(y_true, y_pred, subset_name):

    pearson, spearman = safe_corr(y_true, y_pred)

    return {
        "model": MODEL_NAME,
        "subset": subset_name,
        "n_peptides": len(y_true),
        "R2": r2_score(y_true, y_pred),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "Pearson": pearson,
        "Spearman": spearman,
        "true_coam_mean": np.mean(y_true),
        "pred_coam_mean": np.mean(y_pred),
        "mean_underprediction": np.mean(y_true) - np.mean(y_pred)
    }

eval_rows = []

eval_rows.append(
    evaluate_subset(
        y_test,
        test_pred,
        "Full test set"
    )
)

for q, label in [
    (0.75, "Top 25% true coam"),
    (0.90, "Top 10% true coam"),
    (0.95, "Top 5% true coam")
]:

    threshold = np.quantile(y_test, q)
    mask = y_test >= threshold

    eval_rows.append(
        evaluate_subset(
            y_test[mask],
            test_pred[mask],
            label
        )
    )

eval_df = pd.DataFrame(eval_rows)

print("\nHigh-coam subset evaluation:")
print(eval_df.to_string(index=False))

eval_df.to_csv(
    f"{MODEL_NAME}_high_coam_eval.csv",
    index=False
)

# =========================
# SAVE MODEL
# =========================

model.save_model(
    f"{MODEL_NAME}.json"
)

print(f"\nSaved model: {MODEL_NAME}.json")

# =========================
# FEATURE IMPORTANCE
# =========================

importance_df = pd.DataFrame({
    "model": MODEL_NAME,
    "feature": feature_names,
    "importance": model.feature_importances_
})

importance_df = importance_df.sort_values(
    "importance",
    ascending=False
).reset_index(drop=True)

importance_df.to_csv(
    f"{MODEL_NAME}_feature_importances.csv",
    index=False
)

print("\nTop 30 feature importances:")
print(
    importance_df
    .head(30)
    .to_string(index=False)
)

# =========================
# SAVE PREDICTIONS
# =========================

predictions_df = pd.DataFrame({
    "model": MODEL_NAME,
    SEQ_COLUMN: test[SEQ_COLUMN].values,
    "true_coam": y_test,
    "predicted_coam": test_pred,
    "residual_true_minus_pred": y_test - test_pred
})

predictions_df.to_csv(
    f"{MODEL_NAME}_predictions.csv",
    index=False
)

# =========================
# SAVE SUMMARY
# =========================

summary_df = pd.DataFrame([
    {
        "model": MODEL_NAME,
        "dataset": "train",
        **train_metrics
    },
    {
        "model": MODEL_NAME,
        "dataset": "test",
        **test_metrics
    }
])

summary_df.to_csv(
    f"{MODEL_NAME}_train_test_summary.csv",
    index=False
)

print("\nSaved:")
print(f"{MODEL_NAME}_high_coam_eval.csv")
print(f"{MODEL_NAME}_feature_importances.csv")
print(f"{MODEL_NAME}_predictions.csv")
print(f"{MODEL_NAME}_train_test_summary.csv")

M_onehot20_chemclass5_xgboost_gpu
Feature type: AA one-hot 20 + chemical class one-hot 5
Train shape: (58579, 300)
Test shape : (5119, 300)
AA one-hot features: 240
Chemical class features: 60
Total features: 300
Device: cuda

Train metrics:
R2: 0.7868
MAE: 0.0332
RMSE: 0.0453
Pearson: 0.9113
Spearman: 0.9006

Test metrics:
R2: 0.1512
MAE: 0.0857
RMSE: 0.1114
Pearson: 0.3905
Spearman: 0.3750

High-coam subset evaluation:
                            model            subset  n_peptides         R2      MAE     RMSE  Pearson  Spearman  true_coam_mean  pred_coam_mean  mean_underprediction
M_onehot20_chemclass5_xgboost_gpu     Full test set        5119   0.151189 0.085743 0.111403 0.390486  0.375024        0.313862        0.311936              0.001926
M_onehot20_chemclass5_xgboost_gpu Top 25% true coam        1280  -3.124425 0.126335 0.145988 0.318679  0.295142        0.468338        0.343836              0.124502
M_onehot20_chemclass5_xgboost_gpu Top 10% true coam         512  -7.669441 0.